# Bronze → Silver

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

catalog = "vcredit"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

## Limpeza dos dados

### Verificação dos tipos de dados

In [0]:
df_bronze_chamados = spark.table(f"{bronze_schema}.tb_chamados")

display(df_bronze_chamados.limit(5))

In [0]:
for c in df_bronze_chamados.columns:
    display(
        df_bronze_chamados.select(
            F.count("*").alias(c),
            F.sum(F.col(c).isNull().cast("int")).alias("nulos"),
            F.sum((F.trim(F.col(c)) == "").cast("int")).alias("vazios")
        )
    )

    display(
        df_bronze_chamados.groupBy(c).count()
                          .orderBy(F.desc("count"))
                          .limit(50)
    )

### Tratamento de valores em 'canais', 'motivos' e 'resolvido'

#### Canais

In [0]:
df_bronze_chamados = spark.table(f"{bronze_schema}.tb_chamados")
df_silver_chamados = (
    df_bronze_chamados
        .withColumn("canal", F.lower(F.col("canal")))
        .withColumn("canal", F.regexp_replace(F.col("canal"), r"[^a-z]+", " "))
        .withColumn("canal", F.trim(F.col("canal")))
        .withColumn("canal",
            F.when(F.col("canal").isin("ura", "u r a"), "URA")
            .when(F.col("canal").isin("chatbot", "chat bot"), "Chatbot")
            .when(F.col("canal").isin("atendimento inicial", "atend inicial"), "Atendimento Inicial")
            .when(F.col("canal").isin("atendimento especializado", "atend especializado"), "Atendimento Especializado")
            .otherwise("Não Informado")
    )
)

display(df_silver_chamados.groupBy("canal").count().orderBy(F.desc("count")))


#### Motivos

In [0]:
import unicodedata
from pyspark.sql.types import StringType

@F.udf(StringType())
def remover_acentos(texto):
    if texto is None:
        return None
    texto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

df_bronze_chamados = spark.table(f"{bronze_schema}.tb_chamados")
df_silver_chamados = (
    df_silver_chamados
        .withColumn("motivo", F.lower(F.col("motivo")))
        .withColumn("motivo", remover_acentos(F.col("motivo")))
        .withColumn("motivo", F.regexp_replace(F.col("motivo"), r"[^a-z]+", " "))
        .withColumn("motivo", F.trim(F.col("motivo")))
        .withColumn("motivo",
            F.when(F.col("motivo") == "consulta de fatura", "Consulta de fatura")
            .when(F.col("motivo") == "consulta de limite", "Consulta de limite")
            .when(F.col("motivo") == "consulta de contrato", "Consulta de contrato")
            .when(F.col("motivo") == "contestacao de fatura", "Contestação de fatura")
            .when(F.col("motivo") == "renegociacao de divida", "Renegociação de dívida")
            .when(F.col("motivo") == "bloqueio de cartao", "Bloqueio de cartão")
            .when(F.col("motivo") == "desbloqueio de cartao", "Desbloqueio de cartão")
            .when(F.col("motivo") == "contratacao de cartao adicional", "Contratação de cartão adicional")
            .when(F.col("motivo") == "compra nao autorizada", "Compra não autorizada")
            .when(F.col("motivo") == "alteracao de dados cadastrais vencimento da fatura telefone email", "Alteração de dados cadastrais")
            .when(F.col("motivo") == "transferencia de agencia", "Transferência de agência")
            .when(F.col("motivo") == "cancelamento de conta", "Cancelamento de conta")
            .when(F.col("motivo") == "problema com aplicativo", "Problema com aplicativo")
            .when(F.col("motivo") == "duvidas gerais sobre programa de pontos", "Dúvidas gerais sobre programa de pontos")
            .otherwise("Não Informado")
        )
)

display(df_silver_chamados.groupBy("motivo").count().orderBy(F.desc("count")))


#### Resolvido

In [0]:
df_silver_chamados = (
    df_silver_chamados
        .withColumn("resolvido", F.lower(F.col("resolvido")))
        .withColumn("resolvido", F.trim(F.col("resolvido")))
        .withColumn("resolvido",
            F.when(F.col("resolvido").isin("não", "n", "0", "nao resolvido"), "Não")
            .when(F.col("resolvido").isin("sim", "s", "1"), "Sim")
            .otherwise("Não Informado")
    )
)

display(df_silver_chamados.groupBy("resolvido").count().orderBy(F.desc("count")))


### Linhas duplicadas

#### Verificar dados duplicados

In [0]:
print("linhas:", df_bronze_chamados.count())
print("ids distintos:", df_bronze_chamados.select("id_chamado", "id_cliente", "canal", "motivo", "resolvido").distinct().count())
print("ids duplicados:", df_bronze_chamados.count() - df_bronze_chamados.select("id_chamado", "id_cliente", "canal", "motivo", "resolvido").distinct().count())

#### Remover valores duplicados

In [0]:
print("linhas antes:", df_silver_chamados.count())

df_silver_chamados = (
    df_silver_chamados
    .dropDuplicates(["id_chamado", "id_cliente", "canal", "motivo", "resolvido"])
)

print("linhas depois:", df_silver_chamados.count())

In [0]:
display(df_silver_chamados.limit(10))